Test the connection in the notebook

In [9]:
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

config = dotenv_values(".env")
db_string = config["DB_STRING"]
print("Host:", make_url(db_string).host)

engine = create_engine(db_string)
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM diabetes.patient")).scalar())

Host: ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com
768


In [10]:
import pandas as pd

SCHEMA = "diabetes"
patient          = pd.read_sql_table("patient",          engine, schema=SCHEMA)
blood_metrics    = pd.read_sql_table("blood_metrics",    engine, schema=SCHEMA)
pedigree_outcome = pd.read_sql_table("pedigree_outcome", engine, schema=SCHEMA)
skin             = pd.read_sql_table("skin",             engine, schema=SCHEMA)

In [11]:
import os
os.makedirs("data", exist_ok=True)

patient.to_csv("data/patient.csv", index=False)
blood_metrics.to_csv("data/blood_metrics.csv", index=False)
pedigree_outcome.to_csv("data/pedigree_outcome.csv", index=False)
skin.to_csv("data/skin.csv", index=False)

In [12]:
patient          = pd.read_csv("data/patient.csv")
blood_metrics    = pd.read_csv("data/blood_metrics.csv", parse_dates=["measurement_date"])
pedigree_outcome = pd.read_csv("data/pedigree_outcome.csv")
skin             = pd.read_csv("data/skin.csv")

In [13]:
patient.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           768 non-null    int64  
 1   Age          768 non-null    int64  
 2   pregnancies  768 non-null    int64  
 3   bmi          768 non-null    float64
dtypes: float64(1), int64(3)
memory usage: 24.1 KB


In [14]:
blood_metrics.info()

<class 'pandas.DataFrame'>
RangeIndex: 1771 entries, 0 to 1770
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   insulin           1771 non-null   int64         
 1   glucose           1771 non-null   int64         
 2   bloodpressure     1771 non-null   int64         
 3   patientid         1771 non-null   int64         
 4   id                1771 non-null   int64         
 5   measurement_date  1771 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(5)
memory usage: 83.1 KB


In [15]:
import pandas as pd
from sqlalchemy import text

dataframes = {
    "patient": patient,
    "blood_metrics": blood_metrics,
    "pedigree_outcome": pedigree_outcome,
    "skin": skin,
}

def db_count(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).scalar()

for table, df in dataframes.items():
    t = f'diabetes."{table}"'

    # Rows and columns
    db_rows = db_count(f"SELECT COUNT(*) FROM {t}")
    db_cols = db_count(f"""SELECT COUNT(*) FROM information_schema.columns
                           WHERE table_schema='diabetes' AND table_name='{table}'""")
    print(f"\n===== {table} =====")
    print(f"Rows:    db = {db_rows}   notebook = {df.shape[0]}")
    print(f"Columns: db = {db_cols}   notebook = {df.shape[1]}")

    # NULL/NaN and zeros per column
    rows = []
    for col in df.columns:
        db_null = db_count(f'SELECT COUNT(*) FROM {t} WHERE "{col}" IS NULL')
        nb_null = df[col].isna().sum()

        if pd.api.types.is_numeric_dtype(df[col]):
            db_zero = db_count(f'SELECT COUNT(*) FROM {t} WHERE "{col}" = 0')
            nb_zero = (df[col] == 0).sum()
        else:
            db_zero = nb_zero = "-"   # e.g. the date column

        rows.append([col, db_null, nb_null, db_zero, nb_zero])

    print(pd.DataFrame(rows, columns=["column", "null_db", "null_nb", "zero_db", "zero_nb"])
          .to_string(index=False))


===== patient =====
Rows:    db = 768   notebook = 768
Columns: db = 4   notebook = 4
     column  null_db  null_nb  zero_db  zero_nb
         id        0        0        0        0
        Age        0        0        0        0
pregnancies        0        0      111      111
        bmi        0        0       11       11

===== blood_metrics =====
Rows:    db = 1771   notebook = 1771
Columns: db = 6   notebook = 6
          column  null_db  null_nb zero_db zero_nb
         insulin        0        0     380     380
         glucose        0        0       9       9
   bloodpressure        0        0      36      36
       patientid        0        0       0       0
              id        0        0       0       0
measurement_date        0        0       -       -

===== pedigree_outcome =====
Rows:    db = 768   notebook = 768
Columns: db = 4   notebook = 4
                  column  null_db  null_nb  zero_db  zero_nb
diabetespedigreefunction        0        0        0        0
   

In [16]:
# Patient numbers in each table (without changing the data)
keys = {
    "patient":          patient["id"],
    "blood_metrics":    blood_metrics["patientid"],
    "pedigree_outcome": pedigree_outcome["patientid"],
    "skin":             skin["patientid"],
}

# 1. Overview: rows, different patients, empty IDs, repeated IDs
overview = pd.DataFrame({
    name: {
        "rows":             len(k),
        "unique_patients":  k.nunique(),
        "missing_id (NaN)": k.isna().sum(),
        "repeated_ids":     k.duplicated().sum(),
    }
    for name, k in keys.items()
}).T
print(overview, "\n")

# 2. Compare every table with the patient table
patient_ids = set(patient["id"])
for name, k in keys.items():
    if name == "patient":
        continue
    ids = set(k.dropna())
    not_in_table   = patient_ids - ids   # patients with no row in this table
    not_in_patient = ids - patient_ids   # IDs that don't exist in patient
    print(f"{name}:")
    print(f"  patients missing in this table: {len(not_in_table)}  {sorted(not_in_table)[:10]}")
    print(f"  unknown IDs (not in patient):   {len(not_in_patient)}  {sorted(not_in_patient)[:10]}")

                  rows  unique_patients  missing_id (NaN)  repeated_ids
patient            768              768                 0             0
blood_metrics     1771             1003                 0           768
pedigree_outcome   768              768                 0             0
skin               768              768                 0             0 

blood_metrics:
  patients missing in this table: 0  []
  unknown IDs (not in patient):   235  [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009]
pedigree_outcome:
  patients missing in this table: 0  []
  unknown IDs (not in patient):   0  []
skin:
  patients missing in this table: 0  []
  unknown IDs (not in patient):   0  []


In [17]:
bm = blood_metrics

# How often does each patient ID appear?
counts = bm["patientid"].value_counts()

print("Patient IDs that appear only once:", (counts == 1).sum())
print("Patient IDs that appear more than once:", (counts > 1).sum())
print("Rows belonging to IDs that appear once:", bm["patientid"].isin(counts[counts == 1].index).sum())
print("Rows belonging to IDs that appear more than once:", bm["patientid"].isin(counts[counts > 1].index).sum())

# Only the patients with several rows
rep = bm[bm["patientid"].isin(counts[counts > 1].index)]

# For each column: how many of these patients have DIFFERENT values in their rows?
cols = ["measurement_date", "insulin", "glucose", "bloodpressure"]
print("\nPatients whose rows differ in this column:")
print(rep.groupby("patientid")[cols].nunique(dropna=False).gt(1).sum())

# Are the rows completely identical (all columns except id)?
identical = rep.duplicated(subset=["patientid"] + cols, keep=False)
print("\nRows fully identical to another row of the same patient:", identical.sum())
print("Rows that differ in at least one column:", (~identical).sum())

Patient IDs that appear only once: 235
Patient IDs that appear more than once: 768
Rows belonging to IDs that appear once: 235
Rows belonging to IDs that appear more than once: 1536

Patients whose rows differ in this column:
measurement_date    768
insulin             763
glucose             764
bloodpressure       761
dtype: int64

Rows fully identical to another row of the same patient: 0
Rows that differ in at least one column: 1536


In [19]:
# The 235 IDs that appear only in blood_metrics
unknown_ids = set(blood_metrics["patientid"]) - set(patient["id"])
print("Unknown IDs:", len(unknown_ids), "\n")

# Patient number column in each table
tables = {
    "patient":          patient["id"],
    "pedigree_outcome": pedigree_outcome["patientid"],
    "skin":             skin["patientid"],
    "blood_metrics":    blood_metrics["patientid"],
}

for name, ids in tables.items():
    found = unknown_ids & set(ids)
    print(f"{name:17} → found {len(found):3} of {len(unknown_ids)}")

Unknown IDs: 235 

patient           → found   0 of 235
pedigree_outcome  → found   0 of 235
skin              → found   0 of 235
blood_metrics     → found 235 of 235


In [18]:
# The 235 IDs that appear only in blood_metrics
unknown_ids = set(blood_metrics["patientid"]) - set(patient["id"])
print("Unknown IDs in blood_metrics:", len(unknown_ids))

# Do any of them exist in the outcome table?
in_outcome = unknown_ids & set(pedigree_outcome["patientid"])
print("Of these, found in pedigree_outcome:", len(in_outcome))

# For comparison: the 768 real patients
real_ids = set(patient["id"])
print("Real patients found in pedigree_outcome:",
      len(real_ids & set(pedigree_outcome["patientid"])))

Unknown IDs in blood_metrics: 235
Of these, found in pedigree_outcome: 0
Real patients found in pedigree_outcome: 768


In [20]:
# New table: only patients that exist in the patient table
blood_metrics_clean = blood_metrics[blood_metrics["patientid"].isin(patient["id"])].copy()

# Check
print("Original:", blood_metrics.shape)          # (1771, 6) → unchanged
print("Clean:   ", blood_metrics_clean.shape)    # (1536, 6)
print("Patients in clean:", blood_metrics_clean["patientid"].nunique())   # 768
print("Rows per patient:", blood_metrics_clean.groupby("patientid").size().unique())  # [2]

Original: (1771, 6)
Clean:    (1536, 6)
Patients in clean: 768
Rows per patient: [2]


In [21]:
blood_metrics_unknown = blood_metrics[~blood_metrics["patientid"].isin(patient["id"])].copy()
print("Unknown:", blood_metrics_unknown.shape)   # (235, 6)

Unknown: (235, 6)


In [ ]:
# Backup of the long version (1536 rows, 2 per patient)
blood_metrics_clean_long = blood_metrics_clean.copy()

# 1. Sort by date and number each test: 1 = earlier, 2 = later
bm = blood_metrics_clean_long.sort_values(["patientid", "measurement_date"]).copy()
bm["test_nr"] = bm.groupby("patientid").cumcount() + 1

# 2. Pivot: one row per patient, each whole test moves together
blood_metrics_clean = bm.pivot(index="patientid",
                               columns="test_nr",
                               values=["measurement_date", "insulin", "glucose", "bloodpressure"])

# 3. Column names: ('insulin', 1) → 'insulin_1'
blood_metrics_clean.columns = [f"{col}_{nr}" for col, nr in blood_metrics_clean.columns]
blood_metrics_clean = blood_metrics_clean.reset_index()

# 4. Checks
print("Shape:", blood_metrics_clean.shape)                                  # (768, 9)
print("One row per patient:", blood_metrics_clean["patientid"].is_unique)  # True
print("Empty cells:", blood_metrics_clean.isna().sum().sum())              # 0
print("Date 1 always before date 2:",
      (blood_metrics_clean["measurement_date_1"] < blood_metrics_clean["measurement_date_2"]).all())

check = bm.merge(blood_metrics_clean, on="patientid")
for m in ["measurement_date", "insulin", "glucose", "bloodpressure"]:
    t1 = (check.loc[check["test_nr"] == 1, m] == check.loc[check["test_nr"] == 1, f"{m}_1"]).all()
    t2 = (check.loc[check["test_nr"] == 2, m] == check.loc[check["test_nr"] == 2, f"{m}_2"]).all()
    print(f"{m:17} test 1 correct: {t1}   test 2 correct: {t2}")

# 5. Save
blood_metrics_clean.to_csv("data/blood_metrics_clean.csv", index=False)
blood_metrics_clean.head()